<a href="https://colab.research.google.com/github/GuFerreiraV/notebooks-google-colab/blob/main/Notebook_An%C3%A1lise_de_Emo%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Instalação de dependências



In [63]:
!pip install ultralytics
!pip install deepface
!pip install langchain-ollama
!pip install

ERROR: You must give at least one requirement to install (see "pip help install")


In [64]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 104 not upgraded.
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,919 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,802 kB]
Fetched 12.

### Ollama serve

In [65]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()

time.sleep(10)

#### Baixando modelo tinyllama

In [66]:
!ollama pull tinyllama:1.1b

In [67]:
!ollama list

NAME               ID              SIZE      MODIFIED               
tinyllama:1.1b     2644915ede35    637 MB    Less than a second ago    
llama3.2:latest    a80c4f17acd5    2.0 GB    3 hours ago               


### YOLOv8

In [68]:
from ultralytics import YOLO

# Carrega o modelo
model_yolo_nano = YOLO('yolov8n.pt')

# Realiza a detecção
results = model_yolo_nano.predict(source='images/', conf=0.25)


image 1/2 /content/images/gn-group-jnPVwVPsYLY-unsplash.jpg: 640x480 1 person, 292.1ms
image 2/2 /content/images/land-o-lakes-inc-A5xiXjyxmAU-unsplash.jpg: 448x640 1 person, 1 chair, 1 couch, 164.3ms
Speed: 4.5ms preprocess, 228.2ms inference, 2.9ms postprocess per image at shape (1, 3, 448, 640)


#### Contagem de objetos

In [69]:
from collections import Counter

# Class para "fingir" o objeto que função espera
class ObjectDetected:
    def __init__(self, name, confidence):
        self.name = name
        self.confidence = confidence

# Função para extrair os dados brutos do YOLO e converter para sua estrutura
def formatar_objetos_yolo(resultados_yolo):
    # Passo 1: Filtrar por confiança (nosso limite de 50%)
    objetos_filtrados = [
        obj.name for obj in resultados_yolo
        if obj.confidence >= 0.5
    ]

    # Passo 2: Se a lista estiver vazia, aplicar Redução Dinâmica (30%)
    if not objetos_filtrados:
        objetos_filtrados = [
            obj.name for obj in resultados_yolo
            if obj.confidence >= 0.3
        ]
        status = " (Confiança Reduzida)"
    else:
        status = ""

    if not objetos_filtrados:
        return "Nenhum objeto relevante detectado."

    # Passo 3: Contar ocorrências
    contagem = Counter(objetos_filtrados)

    # Passo 4: Montar a string descritiva
    itens = [f"{qtd} {nome}" for nome, qtd in contagem.items()]
    descricao = ", ".join(itens)

    return f"Objetos detectados{status}: {descricao}."

def extrair_dados_yolo(caminho_imagem, modelo):
    # Roda a detecção (o 'stream=True' é mais eficiente em memória)
    resultados_brutos = modelo.predict(source=caminho_imagem, conf=0.3, verbose=False)

    lista_convertida = []

    for r in resultados_brutos:
        # Pega o dicionário de nomes do modelo (ex: {0: 'person', 1: 'bicycle', ...})
        nomes_classes = r.names

        # Itera pelas caixas (boxes) detectadas
        for box in r.boxes:
            id_classe = int(box.cls[0])    # ID numérico
            conf = float(box.conf[0])      # Confiança (0.0 a 1.0)
            nome = nomes_classes[id_classe] # Transforma ID em texto

            # Cria o objeto que sua função 'formatar_objetos_yolo' entende
            lista_convertida.append(ObjectDetected(nome, conf))

    return lista_convertida

In [70]:
# Carrega o modelo uma única vez no início do script
modelo_nano = YOLO('yolov8n.pt')

# No loop de processamento:
dados_brutos = extrair_dados_yolo("/content/images/", modelo_nano)
string_para_o_prompt = formatar_objetos_yolo(dados_brutos)

print(string_para_o_prompt)
# Saída esperada: "Objetos detectados: 1 person, 2 cups, 1 laptop."

Objetos detectados: 2 person.


## Reconhecimento facial e análise de emoção

In [71]:
from deepface import DeepFace
import cv2

def extrair_emocoes_com_resiliencia(caminho_imagem):
    try:
        # Passo 1: Detecção Inicial
        # O DeepFace retorna uma lista de resultados (um para cada face detectada)
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False # Evita erro se nenhuma face for encontrada
        )

        if not resultados:
            return "Nenhuma expressão facial detectada."

        # Para este exemplo, focaremos na primeira face detectada
        face = resultados[0]
        emocao_detectada = face['dominant_emotion']
        # O DeepFace fornece a pontuação de todas as emoções; pegamos a da dominante
        confianca = face['emotion'][emocao_detectada] / 100

        # Passo 2: Verificação de Limite Fixo (0.5 ou 50%)
        if confianca >= 0.5:
            return f"Emoção: {emocao_detectada} (Alta Confiança: {confianca:.2%})"

        # Passo 3: Redução Dinâmica (Se falhar no passo anterior)
        elif confianca >= 0.3:
            return f"Emoção: {emocao_detectada} (Confiança Reduzida: {confianca:.2%} - Tratar como hipótese)"

        # Passo 4: Falha Crítica
        else:
            return "Nenhuma expressão facial detectada com clareza suficiente."

    except Exception as e:
        return f"Erro no processamento de imagem: {str(e)}"

## Múltiplas Faces

In [72]:
from deepface import DeepFace
from collections import Counter

def analisar_clima_do_grupo(caminho_imagem):
    try:
        # Passo 1: Detecção de todas as faces
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False
        )

        if not resultados or len(resultados) == 0:
            return "Nenhuma expressão facial detectada para análise de grupo."

        emocoes_validas = []
        somas_confianca = 0

        # Passo 2 e 3: Filtragem e Redução Dinâmica por face
        for face in resultados:
            emocao = face['dominant_emotion']
            confianca = face['emotion'][emocao] / 100

            # Aplicamos seu filtro de 50% (ou 30% se necessário)
            if confianca >= 0.3:
                emocoes_validas.append(emocao)
                somas_confianca += confianca

        if not emocoes_validas:
            return "As faces detectadas não possuem clareza suficiente para uma análise segura."

        # Passo 4: Cálculo da Média (Clima do Grupo)
        contagem = Counter(emocoes_validas)
        clima_predominante = contagem.most_common(1)[0][0]
        media_confianca = somas_confianca / len(emocoes_validas)

        # Formatação para o 'Script Ponte'
        status = "Alta Confiança" if media_confianca >= 0.5 else "Confiança Reduzida"
        return f"Clima do Grupo: {clima_predominante} ({status}: {media_confianca:.2%}, baseado em {len(emocoes_validas)} faces)."

    except Exception as e:
        return f"Erro ao processar clima do grupo: {str(e)}"

## Script de Integração

In [73]:
def orquestrar_prompt_final(caminho_instrucoes, dados_yolo, dados_deepface, legenda):
    # 1. Carregar Regras (Fails-Fast se não houver arquivo)
    try:
        with open(caminho_instrucoes, 'r') as f:
            regras = f.read()
            if "### INSTRUÇÕES" not in regras:
                raise ValueError("Tags obrigatórias ausentes no arquivo .md")
    except FileNotFoundError:
        return "ERRO CRÍTICO: Arquivo de instruções não encontrado."

    # 2. Montar o Super Prompt
    prompt = f"{regras}\n\n"
    prompt += "### DADOS DA IMAGEM\n"
    prompt += f"- {dados_yolo}\n"
    prompt += f"- {dados_deepface}\n\n"
    prompt += f"### LEGENDA ORIGINAL\n\"{legenda}\"\n\n"
    prompt += "### ANÁLISE TÉCNICA"

    return prompt

## Função Orquestradora

In [74]:
def montar_super_prompt(caminho_instrucoes, info_yolo, info_deepface, legenda_usuario):
    # 1. Carregamento e Validação das Instruções
    try:
        with open(caminho_instrucoes, 'r') as f:
            instrucoes = f.read()
            # Validação de integridade das tags que definimos
            if "### INSTRUÇÕES" not in instrucoes:
                raise ValueError("Arquivo de instruções inválido: Tag '### INSTRUÇÕES' não encontrada.")
    except FileNotFoundError:
        return "ERRO: O arquivo instructions.md não foi encontrado no caminho especificado."

    # 2. Montagem Estruturada do Prompt
    # Usamos f-strings para injetar as variáveis nas tags correspondentes
    super_prompt = f"{instrucoes}\n\n"
    super_prompt += "### DADOS DA IMAGEM\n"
    super_prompt += f"{info_yolo}\n"
    super_prompt += f"{info_deepface}\n\n"
    super_prompt += f"### LEGENDA ORIGINAL\n\"{legenda_usuario}\"\n\n"
    super_prompt += "### ANÁLISE TÉCNICA\n"

    return super_prompt

In [75]:
import torch
import gc

def limpar_memoria_gpu():
    # 1. Coleta de lixo do Python (limpa referências mortas)
    gc.collect()
    # 2. Limpa o cache do PyTorch na GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [76]:
def gerar_analise_tecnica(prompt_completo, langchain_chain):
    resposta = langchain_chain.invoke({"question": prompt_completo})
    return resposta

## Lógica de extração

In [77]:
import re
import pandas as pd
import os

def extrair_nota(texto_llm):
    # Procura por um número de 0 a 10 no formato "X/10"
    padrao = r"Nota de Dissonância:\s*(\d+)/10"
    match = re.search(padrao, texto_llm)
    if match:
        return int(match.group(1))
    return None # Caso o modelo esqueça de formatar corretamente

def salvar_resultado_csv(dados_nova_analise, arquivo_csv='resultados_tcc.csv'):
    # Se o arquivo já existe, carrega; se não, cria um novo DataFrame
    if os.path.exists(arquivo_csv):
        df = pd.read_csv(arquivo_csv)
    else:
        df = pd.DataFrame(columns=[
            'ID_Imagem', 'Objetos_YOLO', 'Clima_DeepFace',
            'Legenda', 'Analise_LLM', 'Nota_Dissonancia'
        ])

    # Adiciona a nova linha
    df = pd.concat([df, pd.DataFrame([dados_nova_analise])], ignore_index=True)
    df.to_csv(arquivo_csv, index=False)

In [78]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# 1. Configuração do Modelo Local (Ollama)
# Certifique-se de que o 'ollama serve' está rodando em segundo plano
llm_model = OllamaLLM(model="tinyllama:1.1b")

# 2. Definição do Template
template = """Responda à pergunta baseada nas instruções técnicas fornecidas.

Pergunta: {question}

Resposta:"""

prompt_template = ChatPromptTemplate.from_template(template)

# 3. Criação da Chain
chain = prompt_template | llm_model

print("✅ Variável 'chain' inicializada com sucesso!")

✅ Variável 'chain' inicializada com sucesso!


In [79]:
import os
from tqdm import tqdm # Barra de progresso

# --- CONFIGURAÇÕES DO EXPERIMENTO ---
PASTA_IMAGENS = '/content/images/'
ARQUIVO_REGRAS = '/content/instructions.md'
ARQUIVO_SAIDA = '/content/resultados_tcc_final.csv'

# Certifique-se de que a pasta existe
if not os.path.exists(PASTA_IMAGENS):
    print(f"⚠️ Alerta: A pasta {PASTA_IMAGENS} não foi encontrada!")
else:
    # 1. Listar apenas arquivos de imagem (jpg, png, jpeg)
    arquivos = [f for f in os.listdir(PASTA_IMAGENS) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print(f"✅ Iniciando processamento de {len(arquivos)} imagens...")

    # 2. LOOP DE PROCESSAMENTO
    for nome_arquivo in tqdm(arquivos):
        caminho_completo = os.path.join(PASTA_IMAGENS, nome_arquivo)

        try:
            # A. VISÃO COMPUTACIONAL
            # Extraímos os dados brutos e formatamos para o prompt
            dados_brutos_yolo = extrair_dados_yolo(caminho_completo, model_yolo_nano)
            info_yolo = formatar_objetos_yolo(dados_brutos_yolo)
            info_deepface = analisar_clima_do_grupo(caminho_completo)

            # B. LEGENDA (Para o TCC, você pode ter um dict ou ler de um txt)
            # Aqui simulamos uma legenda genérica, mas você pode adaptar
            legenda_teste = "Legenda de teste para análise de dissonância"

            # C. MONTAGEM DO PROMPT E INFERÊNCIA
            super_prompt = montar_super_prompt(ARQUIVO_REGRAS, info_yolo, info_deepface, legenda_teste)

            limpar_memoria_gpu() # 🧹 Essencial para não travar o Colab

            # Chamada da sua chain do LangChain
            resultado_llm = gerar_analise_tecnica(super_prompt, chain)

            # D. EXTRAÇÃO DA NOTA E SALVAMENTO
            nota_extraida = extrair_nota(resultado_llm)

            registro = {
                'ID_Imagem': nome_arquivo,
                'Objetos_YOLO': info_yolo,
                'Clima_DeepFace': info_deepface,
                'Legenda': legenda_teste,
                'Analise_LLM': resultado_llm,
                'Nota_Dissonancia': nota_extraida
            }

            # Salvamento em tempo real (Opção 1 que escolhemos)
            salvar_resultado_csv(registro, ARQUIVO_SAIDA)

        except Exception as e:
            print(f"❌ Erro ao processar {nome_arquivo}: {e}")
            continue

    print(f"\n✨ Processamento concluído! Resultados salvos em: {ARQUIVO_SAIDA}")

✅ Iniciando processamento de 2 imagens...


100%|██████████| 2/2 [1:25:10<00:00, 2555.15s/it]


✨ Processamento concluído! Resultados salvos em: /content/resultados_tcc_final.csv


In [80]:
import pandas as pd
df_teste = pd.read_csv('resultados_tcc_final.csv')
print("--- Primeiros Resultados do TCC ---")
display(df_teste[['ID_Imagem', 'Nota_Dissonancia', 'Clima_DeepFace']])

--- Primeiros Resultados do TCC ---


,ID_Imagem,Nota_Dissonancia,Clima_DeepFace
0,land-o-lakes-inc-A5xiXjyxmAU-unsplash.jpg,NaN,Clima do Grupo: neutral (Alta Confiança: 99.04...
1,gn-group-jnPVwVPsYLY-unsplash.jpg,NaN,"Clima do Grupo: angry (Alta Confiança: 70.68%,..."
